## 08. 打造可配置、可扩展的自动化预处理流水线

PDF 文件 → [PaddleOCR API] → 提取纯文本 → [预处理/分段/元数据] → [Dify API] → 知识库同步  

## 二、关键步骤实现
### 1. PDF 文本提取（PaddleOCR API）
功能 ：调用 PaddleOCR 提取 PDF 中的文本，支持多语言、复杂排版。

In [ ]:
import requests

class OCREngine:
    def __init__(self, config):
        self.api_url = config["ocr"]["api_url"]
        self.high_precision = config["ocr"].get("high_precision", False)
    
    def extract_text(self, file_path):
        """调用 PaddleOCR API 提取 PDF 文本"""
        with open(file_path, "rb") as f:
            files = {"file": f}
            response = requests.post(self.api_url, files=files)
        if response.status_code == 200:
            return response.json().get("text", "")
        else:
            raise Exception(f"OCR 提取失败: {response.text}")

### 2. 文本预处理与分段
功能 ：清洗 OCR 文本并按配置规则分段。
代码示例 ：

In [ ]:
import re

class TextProcessor:
    def __init__(self, config):
        self.config = config
        self.rules = config["dify"]["process_rules"]

    def preprocess(self, text):
        """应用预处理规则（如去空格、去 URL）"""
        if self.rules["pre_processing"][0]["enabled"]:
            text = re.sub(r"\s+", " ", text)  # 去除多余空格 [[7]]
        return text

    def segment(self, text):
        """按配置分段（如按标题分割）"""
        separator = self.rules["segmentation"]["separator"]
        return re.split(separator, text)  # 分段逻辑 [[7]]
